# News Article Classification - Refactor Pipeline

## 1. Project Overview

In this project, we aim to build a Machine Learning model capable of classifying news articles into distinct topics (e.g., Sport, Business, Politics). This is a **Multi-Class Classification** problem. We will transform unstructured text data into numerical features using TF-IDF and train a predictive model to automatically assign categories to unseen articles.

## 🚀 Changes from Experiment to Production

1.  **Pipeline Integration:** Instead of handling Vectorization and Classification as separate steps, we combine them into a single `scikit-learn Pipeline`. This ensures that raw text input is automatically transformed using the exact same logic used during training.
2.  **Artifact Serialization:** We will export a single `.pkl` file containing the entire pipeline, simplifying the REST API code.

In [10]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import os
from datetime import datetime

# Configuration

DATA_PATH = 'bbc-text.csv' #change if there's a new dataset (maybe a more recent one)
MODEL_OUTPUT_PATH = '../app/model/news_classifier.pkl' # Saving directly to the app folder

print("Setup complete.")

# 1. Load Data
try:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset loaded: {df.shape}")
except FileNotFoundError:
    print("Error: csv file not found. Make sure bbc-text.csv is in the notebook folder.")

# 2. Split Data (Raw Text)
# We pass the raw text column 'text' directly to X.
# The cleaning/vectorization will happen INSIDE the pipeline later.
X = df['text']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


# 3. Create the Production Pipeline
# The pipeline acts as a single unit: Input (Text) -> Tfidf -> NaiveBayes -> Output (Label)

production_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('classifier', MultinomialNB())
])

# 4. Train
print("Training pipeline...")
production_pipeline.fit(X_train, y_train)
print("Training complete!")

# 5. Validate Pipeline
y_pred = production_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Current Pipeline Accuracy: {acc:.2%}")

# --- MLOps: Version Control Strategy ---

# Define paths (relative to the notebook folder)
base_model_dir = '../app/model'
version_dir = os.path.join(base_model_dir, 'versions')

# 1. Generate Timestamp for Versioning
# Format: YYYYMMDD_HHMMSS (e.g., 20231027_153000)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
version_filename = f"news_classifier_{timestamp}_acc{acc:.2f}.pkl"
version_path = os.path.join(version_dir, version_filename)

# 2. Define Production Path (Always the same name for the API)
prod_filename = "news_classifier.pkl"
prod_path = os.path.join(base_model_dir, prod_filename)

# 3. Dual Save
print(f"\n--- Saving Artifacts ---")

# A. Save History Version
joblib.dump(production_pipeline, version_path)
print(f"✅ Historical Version saved: {version_path}")

# B. Overwrite Production Version
joblib.dump(production_pipeline, prod_path)
print(f"🚀 Production Model updated: {prod_path}")

print("\nReady for deployment! The API will use the updated 'news_classifier.pkl'.")

Setup complete.
Dataset loaded: (2225, 2)
Training samples: 1780
Test samples: 445
Training pipeline...
Training complete!
Current Pipeline Accuracy: 98.88%

--- Saving Artifacts ---
✅ Historical Version saved: ../app/model/versions/news_classifier_20260108_224123_acc0.99.pkl
🚀 Production Model updated: ../app/model/news_classifier.pkl

Ready for deployment! The API will use the updated 'news_classifier.pkl'.
